In [154]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from scipy.linalg import eigh

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import fastplotlib as fpl

# Init

In [3]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [4]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [5]:
@njit(fastmath=True, cache=True)
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [6]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [7]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-4000])
henon_test_scaled = henon_scaler.transform(henon_dataset[-4000:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [8]:
def henon_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scattergl(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )


    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [9]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [10]:
def generate_henon_grid(all_experiments, cols=3, plot_height=400):
    total_plots = len(all_experiments)
    rows = (total_plots + cols - 1) // cols
    colors = ["white", "magenta"]

    # 1. Initialize the master subplot matrix
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"System #{i+1}" for i in range(total_plots)],
        horizontal_spacing=0.04,
        vertical_spacing=0.03,
    )

    for idx, data_list in enumerate(all_experiments):
        current_row = (idx // cols) + 1
        current_col = (idx % cols) + 1

        for i, data in enumerate(data_list):
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    marker=dict(color=colors[i % len(colors)], size=1),
                    showlegend=False,
                ),
                row=current_row,
                col=current_col,
            )

    fig.update_layout(
        height=plot_height * rows,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=10),
        margin=dict(t=80, b=40, l=40, r=40),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )

    return fig

In [11]:
def create_stiffness_matrix(node_positions, connections):
    num_nodes = node_positions.shape[0]
    dims = node_positions.shape[1]
    K = np.zeros((num_nodes * dims, num_nodes * dims))

    for conn in connections:
        node_conn = conn[:2].astype(int)
        node_pos = node_positions[node_conn]
        k_val = conn[2]

        diff_vec = np.diff(node_pos, axis=0).flatten()
        unit_dir = diff_vec / np.linalg.norm(diff_vec)

        sub_block = np.outer(unit_dir, unit_dir)
        k_local = k_val * np.block([[sub_block, -sub_block], [-sub_block, sub_block]])

        global_indices = (node_conn * dims + np.arange(dims)[:, None]).flatten("F")

        for local_row, global_row in enumerate(global_indices):
            for local_col, global_col in enumerate(global_indices):
                K[global_row, global_col] += k_local[local_row, local_col]

    return K

In [12]:
@njit(fastmath=True, cache=True)
def run_simulation(steps, dt, matrix_size, M_INV, C, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + U[i - 1])

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [13]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=15,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None
):
    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"

    coords = nodes_pos_3d + disp_3d[0]
    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    frame_tracker = 0
    test = True

    def update_springs(canvas):
        nonlocal frame_tracker, test
        # if not test:
        #     return
        # test = False
        frame_tracker = (frame_tracker + frames_moved) % steps

        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]
        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

In [14]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

# Grid

In [195]:
N = 16

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [196]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.3, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.05, 0.3, size=matrix_size))

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = rng.integers(low=0, high=num_nodes, size=(henon_scaled.shape[1]))
# target_nodes = np.array([18, 46])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [197]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_slices = [
    np.s_[:, :-1],
    np.s_[:-1, :],
    np.s_[:-1:2, :-1],
    np.s_[1:-1:2, 1:]
]

dst_slices = [
    np.s_[:, 1:],
    np.s_[1:, :],
    np.s_[1::2, 1:],
    np.s_[2::2, :-1]
]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [ ]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps, 0.05, matrix_size, M_INV, DAMP, K, U
)

X = np.column_stack((displacement, velocity))

In [199]:
X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

x_scaler = StandardScaler()
X_train_scaled, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

Y_train_scaled, Y_test_scaled = (
    henon_train_scaled[transient_steps_reservoir + tau_steps :],
    henon_test_scaled,
)

model = RidgeCV()
model.fit(X_train_scaled, Y_train_scaled)

Y_pred_scaled = model.predict(X_test)
Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.6214 0.3149


In [200]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
henon_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True
).show()

# Modes of Springs

In [14]:
N = 4

nodes_pos = np.arange(N).reshape(-1, 1)
node_ids = np.arange(N)

connections_list = np.column_stack((node_ids[:-1], node_ids[1:], np.ones(N - 1)))
K = create_stiffness_matrix(nodes_pos, connections_list)

M = np.diag(np.ones(N))

eigenvalues, eigenvectors = eigh(K, M)
eigenvalues, eigenvectors

(array([3.28386062e-16, 5.85786438e-01, 2.00000000e+00, 3.41421356e+00]),
 array([[-0.5       ,  0.65328148,  0.5       , -0.27059805],
        [-0.5       ,  0.27059805, -0.5       ,  0.65328148],
        [-0.5       , -0.27059805, -0.5       , -0.65328148],
        [-0.5       , -0.65328148,  0.5       ,  0.27059805]]))

In [15]:
fig = fpl.Figure(canvas="glfw")

node_pos_modes = []
dots = []
for i in range(len(eigenvalues)):
    node_pos_2D = np.column_stack((nodes_pos * 2, np.ones_like(nodes_pos) * -2 * i)).astype(np.float32)
    node_pos_modes.append(node_pos_2D)
    dots.append(fig[0, 0].add_scatter(data=node_pos_2D, sizes=10, colors="magenta"))

step = 0
dt = 0.01

def update_springs(canvas):
    global step
    step += 1
    for i in range(len(eigenvalues)):
        disp = eigenvectors[:, i] * np.sin(np.sqrt(eigenvalues[i]) * step * dt)
        disp_2D = np.column_stack((disp, np.zeros_like(disp))).astype(np.float32)
        coords = node_pos_modes[i] + disp_2D
        dots[i].data[:, :2] = coords.astype(np.float32)


fig.add_animations(update_springs)
fig.show()

In [16]:
t = eigenvectors.T @ M @ eigenvectors
diff = np.eye(len(eigenvalues)) - t
np.linalg.norm(diff, ord="fro")

np.float64(2.1852494449074724e-15)

2026-07-06 10:43:26.299 python[19560:3766280] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-06 10:43:26.299 python[19560:3766280] +[IMKInputSession subclass]: chose IMKInputSession_Modern


# Non Dimensionalized

In [166]:
N = 8

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [167]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.3, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)
M = np.diag(m_diag)

DAMP = np.diag(rng.uniform(0.05, 0.3, size=matrix_size))

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = rng.integers(low=0, high=num_nodes, size=(henon_scaled.shape[1]))
# target_nodes = np.array([18, 46])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [168]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_slices = [np.s_[:, :-1], np.s_[:-1, :], np.s_[:-1:2, :-1], np.s_[1:-1:2, 1:]]

dst_slices = [np.s_[:, 1:], np.s_[1:, :], np.s_[1::2, 1:], np.s_[2::2, :-1]]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [169]:
eigenvalues, eigenvectors = eigh(K, M)
# eigenvalues = np.maximum(eigenvalues, 0) + 1e-12
# eigenvalues[0] = 1e-15
eigenvalues = np.abs(eigenvalues)
eigenvalues[:5]

array([1.03120171e-14, 7.24037596e-15, 8.19335275e-15, 1.23538449e+00,
       1.30223505e+00])

In [170]:
(eigenvectors.T @ M @ eigenvectors).round(6)[:5, :5]

array([[ 1., -0.,  0., -0.,  0.],
       [-0.,  1.,  0.,  0., -0.],
       [ 0.,  0.,  1., -0., -0.],
       [-0.,  0., -0.,  1.,  0.],
       [ 0., -0., -0.,  0.,  1.]])

In [171]:
(eigenvectors.T @ DAMP @ eigenvectors).round(6)[:5, :5]

array([[ 0.984684,  0.065511,  0.017519, -0.086537, -0.030715],
       [ 0.065511,  0.882989,  0.024007, -0.021561, -0.069986],
       [ 0.017519,  0.024007,  0.877115, -0.036952, -0.023654],
       [-0.086537, -0.021561, -0.036952,  1.035505,  0.106513],
       [-0.030715, -0.069986, -0.023654,  0.106513,  0.876805]])

In [172]:
(eigenvectors.T @ K @ eigenvectors).round(6)[:5, :5]

array([[ 0.      ,  0.      ,  0.      , -0.      ,  0.      ],
       [ 0.      ,  0.      , -0.      , -0.      , -0.      ],
       [ 0.      , -0.      ,  0.      , -0.      ,  0.      ],
       [-0.      , -0.      , -0.      ,  1.235384,  0.      ],
       [ 0.      , -0.      ,  0.      ,  0.      ,  1.302235]])

In [173]:
eigenvalues.round(6)[:5]

array([0.      , 0.      , 0.      , 1.235384, 1.302235])

In [174]:
def run_simulation_2(steps, dt, matrix_size, M_INV, DAMP, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    C_modal = eigenvectors.T @ DAMP @ eigenvectors
    F_modal = (eigenvectors.T @ U.T).T
    w_2 = eigenvalues

    for i in range(1, steps):
        acc = F_modal[i - 1] - C_modal @ v[i - 1] - w_2 * x[i - 1]

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = F_modal[i] - C_modal @ v[i - 1] - w_2 * x[i]

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    x = (eigenvectors @ x.T).T
    v = (eigenvectors @ v.T).T

    return x, v

In [175]:
def run_simulation(steps, dt, matrix_size, M_INV, C, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + U[i - 1])

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [182]:
disp, vel = run_simulation_2(
    steps + transient_steps_reservoir + tau_steps, 0.01, matrix_size, M_INV, DAMP, K, U
)
disp.round(2)

array([[ 0.  ,  0.  ,  0.  , ...,  0.  ,  0.  ,  0.  ],
       [ 0.  , -0.  , -0.  , ..., -0.  , -0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  , ..., -0.  ,  0.  ,  0.  ],
       ...,
       [ 0.  , -0.01,  0.  , ..., -0.01,  0.  , -0.  ],
       [ 0.  , -0.01,  0.  , ..., -0.01,  0.  , -0.  ],
       [ 0.  , -0.01,  0.  , ..., -0.01,  0.  , -0.  ]],
      shape=(22001, 128))

In [183]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps, 0.01, matrix_size, M_INV, DAMP, K, U
)
displacement.round(2)

array([[ 0.  ,  0.  ,  0.  , ...,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  , ...,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  , ...,  0.  ,  0.  ,  0.  ],
       ...,
       [ 0.  , -0.01,  0.  , ..., -0.01,  0.  , -0.  ],
       [ 0.  , -0.01,  0.  , ..., -0.01,  0.  , -0.  ],
       [ 0.  , -0.01,  0.  , ..., -0.01,  0.  , -0.  ]],
      shape=(22001, 128))

In [184]:
difference = displacement - disp
distance = np.linalg.norm(difference, ord="fro")
error = distance / np.linalg.norm(displacement, ord="fro")

print(f"Frobenius distance: {distance}")
print(f"Relative error: {error}")

Frobenius distance: 0.17273390025099686
Relative error: 0.0026095319177834365


# Grid Non Dim

In [261]:
N = 20

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [262]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.3, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)
M = np.diag(m_diag)

DAMP = np.diag(rng.uniform(0.05, 0.3, size=matrix_size))

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = rng.integers(low=0, high=num_nodes, size=(henon_scaled.shape[1]))
# target_nodes = np.array([18, 46])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [263]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_slices = [np.s_[:, :-1], np.s_[:-1, :], np.s_[:-1:2, :-1], np.s_[1:-1:2, 1:]]

dst_slices = [np.s_[:, 1:], np.s_[1:, :], np.s_[1::2, 1:], np.s_[2::2, :-1]]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [264]:
eigenvalues, eigenvectors = eigh(K, M)
eigenvalues = np.abs(eigenvalues)

In [271]:
@njit(fastmath=True, cache=True)
def run_simulation_2(steps, dt, matrix_size, DAMP, U, eigenvalues, eigenvectors):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    C_modal = eigenvectors.T @ DAMP @ eigenvectors
    F_modal = (eigenvectors.T @ U.T).T
    w_2 = eigenvalues

    for i in range(1, steps):
        acc = F_modal[i - 1] - C_modal @ v[i - 1] - w_2 * x[i - 1]

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = F_modal[i] - C_modal @ v[i - 1] - w_2 * x[i]

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    x = (eigenvectors @ x.T).T
    v = (eigenvectors @ v.T).T

    return x, v

In [273]:
disp, vel = run_simulation_2(
    steps + transient_steps_reservoir + tau_steps, 0.05, matrix_size, DAMP, U, eigenvalues, eigenvectors
)

In [267]:
X = np.column_stack((disp, vel))
X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

x_scaler = StandardScaler()
X_train_scaled, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

Y_train_scaled, Y_test_scaled = (
    henon_train_scaled[transient_steps_reservoir + tau_steps :],
    henon_test_scaled,
)

model = RidgeCV()
model.fit(X_train_scaled, Y_train_scaled)

Y_pred_scaled = model.predict(X_test)
Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.6210 0.3151


In [256]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
henon_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    disp,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
).show()

# Grid Non Dim removing Matrix Mult

In [139]:
N = 15

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [140]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.3, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)
M = np.diag(m_diag)

DAMP = np.diag(rng.uniform(0.05, 0.3, size=matrix_size))

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = rng.integers(low=0, high=num_nodes, size=(henon_scaled.shape[1]))
# target_nodes = np.array([18, 46])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [141]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_slices = [np.s_[:, :-1], np.s_[:-1, :], np.s_[:-1:2, :-1], np.s_[1:-1:2, 1:]]

dst_slices = [np.s_[:, 1:], np.s_[1:, :], np.s_[1::2, 1:], np.s_[2::2, :-1]]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [142]:
eigenvalues, eigenvectors = eigh(K, M)
eigenvalues = np.abs(eigenvalues)

In [143]:
@njit(fastmath=True, cache=True)
def run_simulation_2(steps, dt, matrix_size, DAMP, U, eigenvalues, eigenvectors):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    C_modal = eigenvectors.T @ DAMP @ eigenvectors
    F_modal = (eigenvectors.T @ U.T).T
    w_2 = eigenvalues

    for i in range(1, steps):
        acc = F_modal[i - 1] - C_modal @ v[i - 1] - w_2 * x[i - 1]

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = F_modal[i] - C_modal @ v[i - 1] - w_2 * x[i]

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    x = (eigenvectors @ x.T).T
    v = (eigenvectors @ v.T).T

    return x, v

In [144]:
@njit(fastmath=True, cache=True)
def run_simulation_2_mult(steps, dt, matrix_size, DAMP, U, eigenvalues, eigenvectors):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    C_modal = eigenvectors.T @ DAMP @ eigenvectors
    C_diag = np.diag(C_modal)
    F_modal = (eigenvectors.T @ U.T).T
    w_2 = eigenvalues

    for i in range(1, steps):
        acc = F_modal[i - 1] - C_diag * v[i - 1] - w_2 * x[i - 1]

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = F_modal[i] - C_diag * v[i - 1] - w_2 * x[i]

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    x = (eigenvectors @ x.T).T
    v = (eigenvectors @ v.T).T

    return x, v, C_modal

In [145]:
disp, vel = run_simulation_2(
    steps + transient_steps_reservoir + tau_steps, 0.05, matrix_size, DAMP, U, eigenvalues, eigenvectors
)

In [146]:
disp_mul, vel_mul, C_modal = run_simulation_2_mult(
    steps + transient_steps_reservoir + tau_steps,
    0.05,
    matrix_size,
    DAMP,
    U,
    eigenvalues,
    eigenvectors,
)

In [148]:
diag_C_modal = np.diag(np.diag(C_modal))
difference = diag_C_modal - C_modal
distance = np.linalg.norm(difference, ord="fro")
error = distance / np.linalg.norm(diag_C_modal, ord="fro")

print(f"Frobenius distance: {distance}")
print(f"Relative error: {error}")

Frobenius distance: 11.215582473107
Relative error: 0.5392428437535173


In [157]:
fig = px.imshow(
    C_modal,
    color_continuous_scale="hot",
    title="Coupling in Modal Damping Matrix",
    labels=dict(color="Magnitude"),  # Colorbar label
)

fig.update_xaxes(side="bottom")
fig.update_traces(
    hovertemplate="Row: %{y}<br>Column: %{x}<br>Value: %{z:<.4f}<extra></extra>"
)

fig.show()

In [149]:
difference = disp - disp_mul
distance = np.linalg.norm(difference, ord="fro")
error = distance / np.linalg.norm(disp, ord="fro")

print(f"Frobenius distance: {distance}")
print(f"Relative error: {error}")

Frobenius distance: 4.584485326583676
Relative error: 0.02558292555044442


In [153]:
difference = vel - vel_mul
distance = np.linalg.norm(difference, ord="fro")
error = distance / np.linalg.norm(vel, ord="fro")

print(f"Frobenius distance: {distance}")
print(f"Relative error: {error}")

Frobenius distance: 31.475923026366917
Relative error: 0.16405058873827016


In [ ]:
X = np.column_stack((disp, vel))
X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

x_scaler = StandardScaler()
X_train_scaled, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

Y_train_scaled, Y_test_scaled = (
    henon_train_scaled[transient_steps_reservoir + tau_steps :],
    henon_test_scaled,
)

model = RidgeCV()
model.fit(X_train_scaled, Y_train_scaled)

Y_pred_scaled = model.predict(X_test)
Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.6214 0.3149


In [ ]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
henon_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    disp,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
).show()

# Non Dimensionalized

In [15]:
N = 8

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [16]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.3, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)
M = np.diag(m_diag)

DAMP = np.diag(rng.uniform(0.05, 0.3, size=matrix_size))

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = rng.integers(low=0, high=num_nodes, size=(henon_scaled.shape[1]))
# target_nodes = np.array([18, 46])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [17]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_slices = [np.s_[:, :-1], np.s_[:-1, :], np.s_[:-1:2, :-1], np.s_[1:-1:2, 1:]]

dst_slices = [np.s_[:, 1:], np.s_[1:, :], np.s_[1::2, 1:], np.s_[2::2, :-1]]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [18]:
k_vals.shape

(161,)

In [306]:
nodes_pos.shape

(64, 2)

In [312]:
node_ids

array([[ 0,  1,  2,  3,  4,  5,  6,  7],
       [ 8,  9, 10, 11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20, 21, 22, 23],
       [24, 25, 26, 27, 28, 29, 30, 31],
       [32, 33, 34, 35, 36, 37, 38, 39],
       [40, 41, 42, 43, 44, 45, 46, 47],
       [48, 49, 50, 51, 52, 53, 54, 55],
       [56, 57, 58, 59, 60, 61, 62, 63]])

In [309]:
K.shape

(128, 128)

In [318]:
A = np.array([
    [1, 0, -1, 0],
    [0, 2, 0, -2],
    [-1, 0, 1, 0],
    [0, -2, 0, 2]
])
B = np.array([1, 2])
C = A / B[:, np.newaxis]
A, B, C

ValueError: operands could not be broadcast together with shapes (4,4) (2,1) 

In [339]:
input = np.array([
    [1],
    [2]
])
input, np.linalg.pinv(input)

(array([[1],
        [2]]),
 array([[0.2, 0.4]]))

In [ ]:
np.linalg.pinv(input) @ K

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 6 is different from 2)

In [113]:
k_inputs = np.array([150, 200])
new_node_conn = np.array([[0, 1, k_inputs[0]], [1, 2, k_inputs[1]]])
new_node_poss = np.array([[0, 0], [1, 0], [2, 0]])
K = create_stiffness_matrix(new_node_poss, new_node_conn)

In [114]:
K_d = np.diag(k_inputs)

C = np.array(
    [
        [-1.0, 0.0, 1.0, 0.0, 0.0, 0.0],  # Spring 1 row
        [0.0, 0.0, -1.0, 0.0, 1.0, 0.0],  # Spring 2 row
    ]
)
K_global = C.T @ K_d @ C

print("Value Matrix (2,2):\n", K_d)
print("\nGlobal Stiffness Matrix (6,6):\n", K_global)

Value Matrix (2,2):
 [[150   0]
 [  0 200]]

Global Stiffness Matrix (6,6):
 [[ 150.    0. -150.    0.    0.    0.]
 [   0.    0.    0.    0.    0.    0.]
 [-150.    0.  350.    0. -200.    0.]
 [   0.    0.    0.    0.    0.    0.]
 [   0.    0. -200.    0.  200.    0.]
 [   0.    0.    0.    0.    0.    0.]]


In [115]:
np.outer(C[0], C[0])

array([[ 1., -0., -1., -0., -0., -0.],
       [-0.,  0.,  0.,  0.,  0.,  0.],
       [-1.,  0.,  1.,  0.,  0.,  0.],
       [-0.,  0.,  0.,  0.,  0.,  0.],
       [-0.,  0.,  0.,  0.,  0.,  0.],
       [-0.,  0.,  0.,  0.,  0.,  0.]])

In [116]:
num_nodes = new_node_poss.shape[0]
dims = new_node_poss.shape[1]
num_springs = new_node_conn.shape[0]
print(f"Num Nodes: {num_nodes}, Dimensions: {dims}, Num Springs: {num_springs}")

K_space = np.zeros((num_springs, num_nodes * dims))
print("K_space:\n", K_space)

for i, conn in enumerate(new_node_conn):
    print()
    print(f"Connectin {i}:")
    node_conn = conn[:2].astype(int)
    print("Node Connection:", node_conn)
    node_pos = new_node_poss[node_conn]
    print("Node Positions:\n", node_pos)
    k_val = conn[2]
    print("Spring Stiffness:", k_val)
    diff_vec = node_pos[1] - node_pos[0]
    print("Displacement Vector:", diff_vec)
    unit_dir = diff_vec / np.linalg.norm(diff_vec)
    print("Unit Direction Vector:", unit_dir)
    sub_block = np.outer(unit_dir, unit_dir)
    print("Sub-Block:\n", sub_block)
    k_local = k_val * np.block([[sub_block, -sub_block], [-sub_block, sub_block]])
    print("Local Stiffness Matrix:\n", k_local)
    global_indices = (node_conn * dims + np.arange(dims)[:, None]).flatten("F")
    print("Global Indices:", global_indices)
    K_space[i, global_indices] = np.tile(unit_dir, dims)
    print("K_space:\n", K_space)

Num Nodes: 3, Dimensions: 2, Num Springs: 2
K_space:
 [[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]

Connectin 0:
Node Connection: [0 1]
Node Positions:
 [[0 0]
 [1 0]]
Spring Stiffness: 150
Displacement Vector: [1 0]
Unit Direction Vector: [1. 0.]
Sub-Block:
 [[1. 0.]
 [0. 0.]]
Local Stiffness Matrix:
 [[ 150.    0. -150.   -0.]
 [   0.    0.   -0.   -0.]
 [-150.   -0.  150.    0.]
 [  -0.   -0.    0.    0.]]
Global Indices: [0 1 2 3]
K_space:
 [[1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]

Connectin 1:
Node Connection: [1 2]
Node Positions:
 [[1 0]
 [2 0]]
Spring Stiffness: 200
Displacement Vector: [1 0]
Unit Direction Vector: [1. 0.]
Sub-Block:
 [[1. 0.]
 [0. 0.]]
Local Stiffness Matrix:
 [[ 200.    0. -200.   -0.]
 [   0.    0.   -0.   -0.]
 [-200.   -0.  200.    0.]
 [  -0.   -0.    0.    0.]]
Global Indices: [2 3 4 5]
K_space:
 [[1. 0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 1. 0.]]


In [117]:
K_space.T @ K_d @ K_space

array([[150.,   0., 150.,   0.,   0.,   0.],
       [  0.,   0.,   0.,   0.,   0.,   0.],
       [150.,   0., 350.,   0., 200.,   0.],
       [  0.,   0.,   0.,   0.,   0.,   0.],
       [  0.,   0., 200.,   0., 200.,   0.],
       [  0.,   0.,   0.,   0.,   0.,   0.]])

In [118]:
k_inputs = np.array([150, 200])
new_node_conn = np.array([[0, 1, k_inputs[0]], [1, 2, k_inputs[1]]])
new_node_poss = np.array([[0, 0], [1, 1], [2, 2]])
K = create_stiffness_matrix(new_node_poss, new_node_conn)
K

array([[  75.,   75.,  -75.,  -75.,    0.,    0.],
       [  75.,   75.,  -75.,  -75.,    0.,    0.],
       [ -75.,  -75.,  175.,  175., -100., -100.],
       [ -75.,  -75.,  175.,  175., -100., -100.],
       [   0.,    0., -100., -100.,  100.,  100.],
       [   0.,    0., -100., -100.,  100.,  100.]])

In [119]:
K_d = np.diag(k_inputs)

val = 1 / np.sqrt(2)

C = np.array(
    [
        [-val, -val, val, val, 0.0, 0.0],  # Spring 1 row
        [0.0, 0.0, -val, -val, val, val],  # Spring 2 row
    ]
)
K_global = C.T @ K_d @ C

print("Value Matrix (2,2):\n", K_d)
print("\nGlobal Stiffness Matrix (6,6):\n", K_global)

Value Matrix (2,2):
 [[150   0]
 [  0 200]]

Global Stiffness Matrix (6,6):
 [[  75.   75.  -75.  -75.    0.    0.]
 [  75.   75.  -75.  -75.    0.    0.]
 [ -75.  -75.  175.  175. -100. -100.]
 [ -75.  -75.  175.  175. -100. -100.]
 [   0.    0. -100. -100.  100.  100.]
 [   0.    0. -100. -100.  100.  100.]]


In [120]:
num_nodes = new_node_poss.shape[0]
dims = new_node_poss.shape[1]
num_springs = new_node_conn.shape[0]
print(f"Num Nodes: {num_nodes}, Dimensions: {dims}, Num Springs: {num_springs}")

K_space = np.zeros((num_springs, num_nodes * dims))
print("K_space:\n", K_space)

for i, conn in enumerate(new_node_conn):
    print()
    print(f"Connectin {i}:")
    node_conn = conn[:2].astype(int)
    print("Node Connection:", node_conn)
    node_pos = new_node_poss[node_conn]
    print("Node Positions:\n", node_pos)
    k_val = conn[2]
    print("Spring Stiffness:", k_val)
    diff_vec = node_pos[1] - node_pos[0]
    print("Displacement Vector:", diff_vec)
    unit_dir = diff_vec / np.linalg.norm(diff_vec)
    print("Unit Direction Vector:", unit_dir)
    global_indices = (node_conn * dims + np.arange(dims)[:, None]).flatten("F")
    print("Global Indices:", global_indices)
    K_space[i, global_indices] = np.concatenate((-unit_dir, unit_dir))
    print("K_space:\n", K_space)

Num Nodes: 3, Dimensions: 2, Num Springs: 2
K_space:
 [[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]

Connectin 0:
Node Connection: [0 1]
Node Positions:
 [[0 0]
 [1 1]]
Spring Stiffness: 150
Displacement Vector: [1 1]
Unit Direction Vector: [0.70710678 0.70710678]
Global Indices: [0 1 2 3]
K_space:
 [[-0.70710678 -0.70710678  0.70710678  0.70710678  0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.        ]]

Connectin 1:
Node Connection: [1 2]
Node Positions:
 [[1 1]
 [2 2]]
Spring Stiffness: 200
Displacement Vector: [1 1]
Unit Direction Vector: [0.70710678 0.70710678]
Global Indices: [2 3 4 5]
K_space:
 [[-0.70710678 -0.70710678  0.70710678  0.70710678  0.          0.        ]
 [ 0.          0.         -0.70710678 -0.70710678  0.70710678  0.70710678]]


In [123]:
new_K = K_space.T @ K_d @ K_space
new_K, K_global - new_K

(array([[  75.,   75.,  -75.,  -75.,    0.,    0.],
        [  75.,   75.,  -75.,  -75.,    0.,    0.],
        [ -75.,  -75.,  175.,  175., -100., -100.],
        [ -75.,  -75.,  175.,  175., -100., -100.],
        [   0.,    0., -100., -100.,  100.,  100.],
        [   0.,    0., -100., -100.,  100.,  100.]]),
 array([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]]))

In [125]:
D = np.array([[0, 0, 1, 1, 1], [1, 0, 0, 0, 0], [0, 1, 0, -1, -2]])  # M  # L  # T

from scipy.linalg import null_space

# Find the basis vectors for the null space
null_vectors = null_space(D)

# Clean up tiny floating point values to see the clean ratios
print(np.round(null_vectors, 4))

[[ 0.      0.    ]
 [ 0.3759  0.7248]
 [-0.5504 -0.1745]
 [ 0.7248 -0.3759]
 [-0.1745  0.5504]]


# True Non Dim

In [162]:
N = 5

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [168]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.3, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

c_diag = rng.uniform(0.05, 0.3, size=matrix_size)
DAMP = np.diag(c_diag)

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = rng.integers(low=0, high=num_nodes, size=(henon_scaled.shape[1]))
# target_nodes = np.array([18, 46])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [164]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_slices = [np.s_[:, :-1], np.s_[:-1, :], np.s_[:-1:2, :-1], np.s_[1:-1:2, 1:]]

dst_slices = [np.s_[:, 1:], np.s_[1:, :], np.s_[1::2, 1:], np.s_[2::2, :-1]]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [165]:
def run_simulation(steps, dt, matrix_size, M_INV, C, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + U[i - 1])

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [ ]:
@njit(fastmath=True, cache=True)
def run_simulation_2(steps, dt, matrix_size, DAMP, U, eigenvalues, eigenvectors):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    C_modal = eigenvectors.T @ DAMP @ eigenvectors
    F_modal = (eigenvectors.T @ U.T).T
    w_2 = eigenvalues

    for i in range(1, steps):
        acc = F_modal[i - 1] - C_modal @ v[i - 1] - w_2 * x[i - 1]

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = F_modal[i] - C_modal @ v[i - 1] - w_2 * x[i]

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    x = (eigenvectors @ x.T).T
    v = (eigenvectors @ v.T).T

    return x, v

In [ ]:
def run_simulation_3(steps, dt, matrix_size, m_diag, c_diag, K_ref, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    zeta = c_diag / (2 * np.sqrt(m_diag * np.diag(K_ref)))

    for i in range(1, steps):
        acc = F_modal[i - 1] - C_modal @ v[i - 1] - w_2 * x[i - 1]

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = F_modal[i] - C_modal @ v[i - 1] - w_2 * x[i]

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    x = (eigenvectors @ x.T).T
    v = (eigenvectors @ v.T).T

    return x, v

Out of the 20 left col do 5 and 5 of each and let the waves do interference basically and have them puhs oto the right.

Also do fourier series to have a smooth inputs